In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import sys
sys.path.append("../../utils/")

from utils import *

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_EXPERIMENTO = "BCCC17__split__v1__pca4_knn__v1"
CARPETA_DATASET = "BCCC17__split__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG KNN =====
N_NEIGHBORS = 5
WEIGHTS = "distance"      # "uniform" o "distance"
METRIC = "minkowski"      # euclidean suele ser minkowski con p=2
P = 2

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 4

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__split__v1__pca4_knn__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(1890959, 64)


,DST_PORT,PROTOCOL,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_SYN_FLAG_COUNTS,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,LABEL
0,443,0,183.413713,3510,21202,4380,0,988.815100,956633.216239,1625.416188,...,1,0,0,5.226951e-02,0.129978,2.692378,8.745763e-02,353.000000,5300.500000,0
1,80,0,172.747578,54,2631,796,0,137.166667,72709.138889,31099.854935,...,1,0,0,3.259388e+00,6.168708,4.541743,7.196825e+00,1.526316,138.473684,0
2,36739,0,0.000000,1,0,0,0,0.000000,0.000000,0.000000,...,0,0,0,1.499345e+09,0.000000,0.000000,1.499345e+09,0.000000,0.000000,0
3,53,1,0.000274,4,76,121,38,79.500000,1722.250000,0.000000,...,0,0,0,9.139000e-05,0.000001,0.000000,2.150000e-06,0.000000,0.000000,0
4,443,0,6.020985,13,599,517,0,60.153846,19067.360947,35175.138889,...,1,0,0,5.017487e-01,1.199572,2.374952,9.996383e-01,6.000000,599.000000,0


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
0,1372038
1,276914
2,129058
3,76583
4,7625
5,6691
6,5485
7,4759
8,4406


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (1890959, 63)
Shape y_train: (1890959,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("knn", KNeighborsClassifier(
        n_neighbors=N_NEIGHBORS,
        weights=WEIGHTS,
        metric=METRIC,
        p=P
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",4
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'fu

In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
scoring = {
    "accuracy": "accuracy",

    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",

    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",

    "mcc": make_scorer(matthews_corrcoef)
}

In [11]:
cv_results = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

cv_results.keys()

dict_keys(['fit_time', 'score_time', 'test_accuracy', 'test_precision_weighted', 'test_recall_weighted', 'test_f1_weighted', 'test_precision_macro', 'test_recall_macro', 'test_f1_macro', 'test_mcc'])

In [12]:
df_folds = pd.DataFrame({
    "fold": np.arange(1, N_SPLITS + 1),

    "accuracy": cv_results["test_accuracy"],

    "precision_weighted": cv_results["test_precision_weighted"],
    "recall_weighted": cv_results["test_recall_weighted"],
    "f1_weighted": cv_results["test_f1_weighted"],

    "precision_macro": cv_results["test_precision_macro"],
    "recall_macro": cv_results["test_recall_macro"],
    "f1_macro": cv_results["test_f1_macro"],

    "mcc": cv_results["test_mcc"],

    "fit_time": cv_results["fit_time"],
    "score_time": cv_results["score_time"]
})

df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,fit_time,score_time
0,1,0.995962,0.995936,0.995962,0.995938,0.935266,0.891799,0.906955,0.990944,3.915865,4.120911
1,2,0.995886,0.995851,0.995886,0.995856,0.877785,0.893144,0.882643,0.990771,3.803834,4.517137
2,3,0.995894,0.995882,0.995894,0.995878,0.903687,0.890615,0.896004,0.990792,4.262815,4.973828
3,4,0.995658,0.995615,0.995658,0.995620,0.887002,0.885623,0.885382,0.990260,4.242928,5.245408
4,5,0.995939,0.995914,0.995939,0.995913,0.890738,0.928589,0.905669,0.990890,5.930782,5.131083


In [13]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_train": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_train": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_neighbors": N_NEIGHBORS,
        "weights": WEIGHTS,
        "metric": METRIC,
        "p": P,
        "n_components_pca": N_COMPONENTS_PCA
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'BCCC17__split__v1__pca4_knn__v1',
 'dataset_train': '/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__train.csv',
 'shape_train': {'rows': 1890959, 'cols': 64},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'n_neighbors': 5,
  'weights': 'distance',
  'metric': 'minkowski',
  'p': 2,
  'n_components_pca': 4},
 'metricas_media': {'accuracy': 0.9958677052600585,
  'precision_weighted': 0.9958394824548605,
  'recall_weighted': 0.9958677052600585,
  'f1_weighted': 0.9958409367574422,
  'precision_macro': 0.898895678945211,
  'recall_macro': 0.8979538622913357,
  'f1_macro': 0.8953305369501502,
  'mcc': 0.9907312967918802,
  'fit_time': 4.431244659423828,
  'score_time': 4.79767336845398},
 'metricas_std': {'accuracy': 0.00012127574417353773,
  'precision_weighted': 0.00012954645155648195,
  'recall_weighted': 0.00012127574417353773,
  'f1_weighted': 0.0001276334141531936,
  'prec

In [14]:
print("========== RESULTADOS CV ==========")
print(f"Accuracy            : {summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}")
print()

print(f"Precision weighted  : {summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}")
print(f"Recall weighted     : {summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}")
print(f"F1 weighted         : {summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}")
print(f"Recall macro        : {summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}")
print(f"F1 macro            : {summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}")
print()

print(f"MCC                 : {summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}")
print()
print(f"Fit time medio      : {summary_cv['metricas_media']['fit_time']:.6f}")
print(f"Score time medio    : {summary_cv['metricas_media']['score_time']:.6f}")

========== RESULTADOS CV ==========
Accuracy            : 0.995868 ± 0.000121

Precision weighted  : 0.995839 ± 0.000130
Recall weighted     : 0.995868 ± 0.000121
F1 weighted         : 0.995841 ± 0.000128

Precision macro     : 0.898896 ± 0.022358
Recall macro        : 0.897954 ± 0.017360
F1 macro            : 0.895331 ± 0.011207

MCC                 : 0.990731 ± 0.000273

Fit time medio      : 4.431245
Score time medio    : 4.797673


In [15]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__split__v1__pca4_knn__v1/BCCC17__split__v1__pca4_knn__v1__folds.csv


In [16]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__split__v1__pca4_knn__v1/BCCC17__split__v1__pca4_knn__v1__summary_cv.json


In [17]:
df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,fit_time,score_time
0,1,0.995962,0.995936,0.995962,0.995938,0.935266,0.891799,0.906955,0.990944,3.915865,4.120911
1,2,0.995886,0.995851,0.995886,0.995856,0.877785,0.893144,0.882643,0.990771,3.803834,4.517137
2,3,0.995894,0.995882,0.995894,0.995878,0.903687,0.890615,0.896004,0.990792,4.262815,4.973828
3,4,0.995658,0.995615,0.995658,0.995620,0.887002,0.885623,0.885382,0.990260,4.242928,5.245408
4,5,0.995939,0.995914,0.995939,0.995913,0.890738,0.928589,0.905669,0.990890,5.930782,5.131083


In [18]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(472740, 64)


,DST_PORT,PROTOCOL,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_SYN_FLAG_COUNTS,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,LABEL
0,443,0,0.000000,1,0,0,0,0.0,0.00,0.0,...,0,0,0,1.499195e+09,1.499195e+09,0.0,0.000000e+00,0.0,0.0,0
1,53,1,0.741115,4,62,61,31,46.0,225.00,0.0,...,0,0,0,2.470384e-01,2.860000e-06,0.0,4.721000e-05,0.0,0.0,0
2,53,1,0.024214,4,82,124,41,82.5,1722.25,0.0,...,0,0,0,8.071340e-03,3.100000e-06,0.0,3.100000e-06,0.0,0.0,0
3,3737,0,0.000048,2,0,0,0,0.0,0.00,0.0,...,0,0,1,4.816000e-05,1.499451e+09,0.0,1.499451e+09,0.0,0.0,2
4,389,0,0.000048,2,0,0,0,0.0,0.00,0.0,...,0,0,0,4.792000e-05,1.499436e+09,0.0,1.499436e+09,0.0,0.0,0


In [19]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,343010
1,69229
2,32265
3,19146
4,1906
5,1673
6,1371
7,1190
8,1102


In [20]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (472740, 63)
Shape y_test: (472740,)


In [21]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [22]:
pipeline.fit(X_train, y_train)

print("Modelo final entrenado con todo el dataset train.")

Modelo final entrenado con todo el dataset train.


In [23]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

Predicciones en test generadas.
Número de predicciones: 472740


In [24]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

{'accuracy': 0.9960781825104709,
 'precision_weighted': 0.9960423733531119,
 'recall_weighted': 0.9960781825104709,
 'f1_weighted': 0.9960507409858389,
 'precision_macro': 0.8985340509682423,
 'recall_macro': 0.8508604496263896,
 'f1_macro': 0.8681313772243674,
 'mcc': 0.9912016023074409}

In [25]:
print("========== RESULTADOS TEST ==========")
print(f"Accuracy            : {metricas_test['accuracy']:.6f}")
print()

print(f"Precision weighted  : {metricas_test['precision_weighted']:.6f}")
print(f"Recall weighted     : {metricas_test['recall_weighted']:.6f}")
print(f"F1 weighted         : {metricas_test['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {metricas_test['precision_macro']:.6f}")
print(f"Recall macro        : {metricas_test['recall_macro']:.6f}")
print(f"F1 macro            : {metricas_test['f1_macro']:.6f}")
print()

print(f"MCC                 : {metricas_test['mcc']:.6f}")

========== RESULTADOS TEST ==========
Accuracy            : 0.996078

Precision weighted  : 0.996042
Recall weighted     : 0.996078
F1 weighted         : 0.996051

Precision macro     : 0.898534
Recall macro        : 0.850860
F1 macro            : 0.868131

MCC                 : 0.991202


In [26]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,342252,189,33,7,210,76,57,135,0,26,15,8,2,0
1,76,69132,0,7,0,10,4,0,0,0,0,0,0,0
2,39,1,32202,0,14,5,2,0,0,1,0,1,0,0
3,2,7,0,19134,0,3,0,0,0,0,0,0,0,0
4,348,0,6,0,1552,0,0,0,0,0,0,0,0,0
5,102,65,0,3,0,1484,13,0,0,6,0,0,0,0
6,32,1,0,0,0,12,1303,0,0,23,0,0,0,0
7,52,0,0,0,4,1,0,1133,0,0,0,0,0,0
8,5,0,0,0,0,0,0,0,1097,0,0,0,0,0
9,20,1,0,0,0,6,16,0,0,981,0,0,0,0


In [27]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    343010
           1       1.00      1.00      1.00     69229
           2       1.00      1.00      1.00     32265
           3       1.00      1.00      1.00     19146
           4       0.87      0.81      0.84      1906
           5       0.93      0.89      0.91      1673
           6       0.93      0.95      0.94      1371
           7       0.89      0.95      0.92      1190
           8       1.00      1.00      1.00      1102
           9       0.95      0.96      0.95      1024
          10       0.82      0.79      0.81       546
          11       0.69      0.67      0.68       271
          12       0.50      0.40      0.44         5
          13       1.00      0.50      0.67         2

    accuracy                           1.00    472740
   macro avg       0.90      0.85      0.87    472740
weighted avg       1.00      1.00      1.00    472740



In [28]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_neighbors": N_NEIGHBORS,
        "weights": WEIGHTS,
        "metric": METRIC,
        "p": P,
        "n_components_pca": N_COMPONENTS_PCA
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "mcc": float(metricas_test["mcc"])
    }
}

summary_test

{'experimento': 'BCCC17__split__v1__pca4_knn__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/BCCC17__split__v1/BCCC17__split__v1__test.csv',
 'shape_test': {'rows': 472740, 'cols': 64},
 'parametros': {'label_col': 'LABEL',
  'n_neighbors': 5,
  'weights': 'distance',
  'metric': 'minkowski',
  'p': 2,
  'n_components_pca': 4},
 'metricas_test': {'accuracy': 0.9960781825104709,
  'precision_weighted': 0.9960423733531119,
  'recall_weighted': 0.9960781825104709,
  'f1_weighted': 0.9960507409858389,
  'precision_macro': 0.8985340509682423,
  'recall_macro': 0.8508604496263896,
  'f1_macro': 0.8681313772243674,
  'mcc': 0.9912016023074409}}

In [29]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__split__v1__pca4_knn__v1/BCCC17__split__v1__pca4_knn__v1__metricas_test.csv


In [30]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__split__v1__pca4_knn__v1/BCCC17__split__v1__pca4_knn__v1__confusion_matrix_test.csv


In [31]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/BCCC17__split__v1__pca4_knn__v1/BCCC17__split__v1__pca4_knn__v1__summary_test.json


In [32]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.9958677052600585, 'precision_weighted': 0.9958394824548605, 'recall_weighted': 0.9958677052600585, 'f1_weighted': 0.9958409367574422, 'precision_macro': 0.898895678945211, 'recall_macro': 0.8979538622913357, 'f1_macro': 0.8953305369501502, 'mcc': 0.9907312967918802, 'fit_time': 4.431244659423828, 'score_time': 4.79767336845398}

TEST:
{'accuracy': 0.9960781825104709, 'precision_weighted': 0.9960423733531119, 'recall_weighted': 0.9960781825104709, 'f1_weighted': 0.9960507409858389, 'precision_macro': 0.8985340509682423, 'recall_macro': 0.8508604496263896, 'f1_macro': 0.8681313772243674, 'mcc': 0.9912016023074409}
